## 🤖 Reddit Collection -- r/Colombia Political Discourse

### By:
MGO

### Date:
2026-03-22

### Description:

Collect political discussion posts and comments from r/Colombia (500,000--700,000 members).
Reddit is a key source because it captures grassroots political discussion -- unlike news
outlets, the discourse here reflects everyday citizen sentiment, including Colombian slang,
memes, and unfiltered political opinions.

**API:** Reddit API via PRAW (Python Reddit API Wrapper)

**Requires:** `REDDIT_CLIENT_ID`, `REDDIT_CLIENT_SECRET` in `.env`

**Data source priority:** P0 (Day 1)

**Output:** `data/01_raw/reddit/reddit_posts_YYYYMMDD.json`

## 📚 Import libraries

In [ ]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Add src to path
sys.path.insert(0, str(Path("../../src").resolve()))

from utils.config import load_config
from utils.normalize import normalize_post

# Load environment variables
load_dotenv(Path("../../.env"))

## ⚙️ Configuration

In [ ]:
# Load Reddit configuration
config = load_config("../../conf/data_collection/reddit.yml")
subreddits = config["subreddits"]
search_keywords = config["search_keywords"]
settings = config["settings"]

print(f"Target subreddits: {[s['name'] for s in subreddits]}")
print(f"Search keywords: {search_keywords}")
print(f"Max posts per sort: {settings['max_posts_per_sort']}")

# Check for API credentials
# TODO: Set REDDIT_CLIENT_ID and REDDIT_CLIENT_SECRET in .env
has_credentials = bool(os.getenv("REDDIT_CLIENT_ID"))
print(f"\nReddit API credentials configured: {has_credentials}")
if not has_credentials:
    print("WARNING: Set Reddit credentials in .env to collect live data")

## 💾 Data Collection

Collect posts using three strategies:
1. **Hot posts** -- currently trending content (captures what people are engaging with now)
2. **New posts** -- most recent content (captures breaking discussions)
3. **Keyword search** -- posts matching election/political keywords from config

In [ ]:
# TODO: Replace with live collection when API credentials are configured
# Uncomment and run when REDDIT_CLIENT_ID and REDDIT_CLIENT_SECRET are set:
#
# from collectors.reddit import collect_reddit_posts
#
# all_posts = []
# for sub_config in subreddits:
#     subreddit = sub_config["name"]
#
#     # Strategy 1: Hot posts
#     hot_posts = collect_reddit_posts(subreddit, sort="hot", limit=settings["max_posts_per_sort"])
#     all_posts.extend(hot_posts)
#     print(f"Collected {len(hot_posts)} hot posts from r/{subreddit}")
#
#     # Strategy 2: New posts
#     new_posts = collect_reddit_posts(subreddit, sort="new", limit=settings["max_posts_per_sort"])
#     all_posts.extend(new_posts)
#     print(f"Collected {len(new_posts)} new posts from r/{subreddit}")
#
#     # Strategy 3: Keyword search
#     search_posts = collect_reddit_posts(subreddit, keywords=search_keywords, limit=settings["max_posts_per_sort"])
#     all_posts.extend(search_posts)
#     print(f"Collected {len(search_posts)} keyword-matched posts from r/{subreddit}")

# For now, demonstrate with sample data
print("Using sample data (set Reddit API credentials for live collection)")
sample_path = Path("../../data/sample/sample_posts.json")
with open(sample_path) as f:
    all_sample = json.load(f)
all_posts = [p for p in all_sample if p["platform"] == "reddit"]
print(f"Loaded {len(all_posts)} sample Reddit posts")

In [ ]:
# Convert to DataFrame
df_raw = pd.DataFrame(all_posts)
print(f"Shape: {df_raw.shape}")
df_raw.head()

## 👷 Data Normalization

In [ ]:
# Normalize to common schema
normalized = [normalize_post(post, source="reddit") for post in all_posts]
df_normalized = pd.DataFrame(normalized)
print(f"Normalized {len(df_normalized)} posts to common schema")
df_normalized[["id", "text", "source", "platform", "author"]].head()

## 💾 Save to Raw Layer

In [ ]:
# Save raw posts
today = datetime.now().strftime("%Y%m%d")
output_dir = Path("../../data/01_raw/reddit")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / f"reddit_posts_{today}.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_posts, f, ensure_ascii=False, indent=2, default=str)

print(f"Saved {len(all_posts)} posts to {output_path}")

## 📊 Collection Summary

In [ ]:
print("=" * 60)
print("REDDIT COLLECTION SUMMARY")
print("=" * 60)
print(f"Total posts collected: {len(all_posts)}")
print(f"Unique authors: {df_raw['author'].nunique()}")
if "score" in df_raw.columns:
    print(f"Score range: {df_raw['score'].min()} to {df_raw['score'].max()}")
print("\nSample posts:")
for _, row in df_raw.head(3).iterrows():
    text = row.get("text", row.get("title", ""))[:80]
    print(f"  [{row.get('author', 'unknown')}] {text}...")

## 💡 Notes & Next Steps

**Quality observations:**
- r/Colombia has very active political discussion, especially during election season
- Posts contain significant Colombian slang that needs preprocessing for NLP
- Score (upvotes - downvotes) is a useful engagement proxy
- Some deleted/removed posts may have empty selftext

**Research notes:**
- Reddit is one of the few platforms where Colombian political discourse is publicly accessible
- The platform has documented issues with organized voting brigades (bodegas)
- Political flairs help pre-categorize content

**Next steps:**
1. Deduplicate posts collected via different sort modes
2. Language detection to filter non-Spanish posts
3. Slang normalization before sentiment analysis

## 📖 References

- PRAW documentation: https://praw.readthedocs.io/
- Reddit API terms: https://www.reddit.com/wiki/api
- r/Colombia subreddit: https://www.reddit.com/r/Colombia/